# L10-1 人工把关：保存会商稿前必须确认

<!-- course-meta: real-llm -->

气象服务对外发布有责任要求。“保存/发布会商稿”“发送通知”这类工具，
智能体执行前必须暂停，由人确认、修改或拒绝。

deepagents 用 `interrupt_on` 指定哪些工具要人工确认；需要 `checkpointer` 记住暂停时的状态。

运行方式：在 VS Code 或 Jupyter 中逐格运行（Shift+Enter）。这一讲要用**真实模型**，
`.env` 里的 `MAIN_API_KEY` 必须填好。
暂时没有 Key，就先看 `99_假模型剧本.ipynb`：那里用预设剧本演示同样的流程。

In [ ]:
# 改了 common/、weather_tools/ 里的代码后自动重新加载，不用重启内核
%load_ext autoreload
%autoreload 2

import uuid

from deepagents import create_deep_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from common import llm
from common.paths import OUTPUT_DIR
from common.trace import show


@tool
def save_report(title: str, content: str) -> str:
    """把会商稿保存为 Markdown 文件。保存前需人工确认。"""
    OUTPUT_DIR.mkdir(exist_ok=True)
    path = OUTPUT_DIR / f"{title}.md"
    path.write_text(content, encoding="utf-8")
    return f"已保存到 {path.name}"


def build(model):
    return create_deep_agent(
        model=model,
        tools=[save_report],
        system_prompt=("你是会商助手。写好会商稿后调用 save_report 保存。"
                       "人工审核时可以修改标题和正文；审核后的参数代表用户的最终决定，"
                       "不应视为错误、未授权或需要重新保存，也不要重新发起原调用。"
                       "以 save_report 的实际工具返回为准，准确说明保存结果；"
                       "不要用其他文件检索工具核验保存结果。用户拒绝保存时如实告知未保存。"),
        interrupt_on={"save_report": {"allowed_decisions": ["approve", "edit", "reject"]}},
        checkpointer=InMemorySaver(),
    )


DRAFT = "# 早间会商\n过去 24 小时无活动台风……"

## 1. 启动交互式人工审核

推荐在课程根目录的终端运行独立脚本（需要配置 `MAIN_API_KEY`）：

```powershell
uv run python .\lessons\L10_人工把关与评测\hitl_cli.py
```

没有 Key 时，可加 `--fake` 使用假模型练习；保存操作仍会真正写入文件：

```powershell
uv run python .\lessons\L10_人工把关与评测\hitl_cli.py --fake
```

也可以运行下方 notebook 单元格：输入框能否正常使用取决于 notebook 环境。选择 `a` 批准、`e` 修改或 `r` 拒绝。修改正文时逐行输入，单独输入 `.` 结束；直接输入 `.` 保留原文。每次运行创建新的会话。

In [ ]:
def ask_decision(request):
    """展示一个待执行操作，并通过输入获取人工决定。"""
    print(f"\n待审核工具：{request['name']}")
    print("待审核参数：")
    for key, value in request["args"].items():
        print(f"  {key}:\n{value}" if key == "content" else f"  {key}: {value}")

    while True:
        choice = input("请选择 [a] 批准 / [e] 修改 / [r] 拒绝：").strip().lower()
        if choice == "a":
            return {"type": "approve"}
        if choice == "r":
            reason = input("拒绝原因（可留空）：").strip() or "人工审核未通过"
            return {"type": "reject", "message": reason}
        if choice == "e":
            if request["name"] != "save_report":
                print("该示例只支持修改 save_report 的参数，请选择批准或拒绝。")
                continue
            args = dict(request["args"])
            title = input(f"新标题（直接回车保留 {args['title']}）：").strip()
            if title:
                args["title"] = title
            print("请输入新的正文，每行回车；单独输入 . 结束，直接输入 . 保留原文：")
            lines = []
            while True:
                line = input()
                if line == ".":
                    break
                lines.append(line)
            if lines:
                args["content"] = "\n".join(lines)
            print(f"修改后的标题：{args['title']}\n修改后的正文：\n{args['content']}")
            confirm = input("按修改后的内容执行？[y/N]：").strip().lower()
            if confirm == "y":
                return {"type": "edit", "edited_action": {"name": request["name"], "args": args}}
            print("未确认修改，请重新选择。")
            continue
        print("无效选择，请输入 a、e 或 r。")


def run_hitl(agent, user_request):
    """启动独立会话，循环处理暂停，直到智能体完成。"""
    cfg = {"configurable": {"thread_id": str(uuid.uuid4())}}
    result = agent.invoke({"messages": [{"role": "user", "content": user_request}]}, config=cfg)
    while result.get("__interrupt__"):
        interrupts = result["__interrupt__"]
        if len(interrupts) != 1:
            raise RuntimeError(f"遇到 {len(interrupts)} 个并发中断，本示例无法安全处理。")
        requests = interrupts[0].value["action_requests"]
        decisions = [ask_decision(request) for request in requests]
        result = agent.invoke(Command(resume={"decisions": decisions}), config=cfg)
    print("\n智能体运行结果：")
    show(result, max_tool_chars=500)
    return result


agent = build(llm.main_llm())
result = run_hitl(agent, "把这份会商稿保存下来：" + DRAFT)

## 2. 观察实际结果

审核批准或修改后，`save_report` 才会写入文件；拒绝时该次调用不会执行。请查看上面的工具返回，并检查 `outputs/` 中的文件内容。目录可能包含之前练习留下的文件，不能仅凭文件存在判断本次是否保存。

In [ ]:
print("现有会商稿文件：", sorted(p.name for p in OUTPUT_DIR.glob("早间会商*.md")))

## 3. 再试一次

重新运行交互单元格，分别尝试批准、修改和拒绝。比较工具返回与模型最终回答；如果模型再次发起保存请求，程序会再次暂停并等待审核。